# Chapter-4: Generating Structured Outputs

# <font color="#800000">LangChain With Structured Output Method</font>

## <font color="#3366FF">TypedDict Schema</font>

### Example-1

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import TypedDict, List

## Load OpenAI API key from the env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Create OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define Expected Output Format
class UserProfile(TypedDict):
    name: str
    age: int
    location: str
    interests: List[str]

## Use with_structured_output
llm_with_structured_output = llm.with_structured_output(UserProfile)

## Invoke LLM
llm_with_structured_output.invoke("I'm Priya, 44, live in Bangalore and I love yoga, travel, data science.")


{'name': 'Priya',
 'age': 44,
 'location': 'Bangalore',
 'interests': ['yoga', 'travel', 'data science']}

### Example-2

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import TypedDict, List, Optional, Annotated

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define a custom class UserProfile using TypedDict
class UserProfile(TypedDict):
    name: Annotated[str, "The user's full name. It should be converted to uppercase"]
    age: Annotated[int, "The user's age. If Age is below 18 append (Underage) near name field."]
    location: Annotated[str, "The user's location. Append country after comma if not provided"]
    interests: Annotated[Optional[List[str]], "The user's interest. Exclude interests which are related to studies"]


## Use with_structured_output
llm_with_structured_output = llm.with_structured_output(UserProfile)

## Invoke LLM and Print output
output = llm_with_structured_output.invoke("I'm Annie, 16, live in London, and I like painting, reading comics.. and maths.")
print(output)


{'name': 'Annie', 'age': 16, 'location': 'London', 'interests': ['painting', 'reading comics', 'maths']}


#### What About JSON, YAML, XML, etc.?

In [4]:
#Check type of the output
type(output)


dict

##### JSON Format

In [5]:
import json

# Convert to JSON
json_output = json.dumps(output, indent=2)
print(json_output)


{
  "name": "Annie",
  "age": 16,
  "location": "London",
  "interests": [
    "painting",
    "reading comics",
    "maths"
  ]
}


##### YAML Format

In [6]:
import yaml

# Convert to YAML
yaml_output = yaml.dump(output, sort_keys=False)
print(yaml_output)


name: Annie
age: 16
location: London
interests:
- painting
- reading comics
- maths



##### XML Format

In [7]:
from dicttoxml import dicttoxml

# dicttoxml expects the root key to be added, or it uses 'root' by default
xml_output = dicttoxml(output, custom_root='user', attr_type=False).decode()
print(xml_output)


<?xml version="1.0" encoding="UTF-8" ?><user><name>Annie</name><age>16</age><location>London</location><interests><item>painting</item><item>reading comics</item><item>maths</item></interests></user>


##### Python List

In [8]:
# Convert to List
list_output = list(output.values())
print(list_output)


['Annie', 16, 'London', ['painting', 'reading comics', 'maths']]


## <font color="#3366FF">Pydantic BaseModel Schema</font>

### Example

In [9]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List, Optional, Annotated

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define a custom class UserProfile using TypedDict
class UserProfile(BaseModel):
    name: str
    age: Annotated[int, Field(gt=18)]
    location: str
    interests: Optional[List[str]] = None

## Use with_structured_output
llm_with_structured_output = llm.with_structured_output(UserProfile)

## Invoke LLM
output = llm_with_structured_output.invoke("John here.. 31, from Ohio I am into jazz music, physics and camping")

## Print Output as Dictionary
print(output.model_dump())


{'name': 'John', 'age': 31, 'location': 'Ohio', 'interests': ['jazz music', 'physics', 'camping']}


## <font color="#3366FF">JSON Schema</font>

### Example

In [10]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

# Defining schema as a plain JSON Schema dict
user_profile_schema = {
    "name": "extract_user_profile",
    "description": "Extracts user profile details such as name, age, location, and interests.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string", 
                     "description": "The user's full name. It should be converted to uppercase"
                    },
            "age": {
                "type": "integer",
                "description": "This is user's age. If Age is below 18 append (Underage) near name field."
            },
            "location": {"type": "string", 
                         "description": "The user's location. Append country after comma if not provided"
                        },
            "interests": {
                "type": "array",
                "items": {"type": "string"},
                "description": "The user's interest. Exclude interests which are related to studies"
            },
        },
        "required": ["name", "age", "location"]
    }
}

# Use with_structured_output
llm_with_structured_output = llm.with_structured_output(user_profile_schema)

# Invoke LLM
llm_with_structured_output.invoke("I'm Annie, 16, live in London, and I like painting, reading comics.. and maths.")


{'name': 'ANNIE (Underage)',
 'age': 16,
 'location': 'London, UK',
 'interests': ['painting', 'reading comics']}

## <font color="#3366FF">More Use Cases</font>

### Example-1

In [11]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import TypedDict

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define Expected Output Format
class Topic(TypedDict):
    fact: str
    joke: str

## Use with_structured_output
llm_with_structured_output = llm.with_structured_output(Topic)

## Invoke LLM
llm_with_structured_output.invoke("ice")


{'fact': 'Ice is the solid form of water; its most common crystalline form (hexagonal ice) is less dense than liquid water, so ice floats. Water expands as it freezes, which is why ice can crack pipes and split rocks over time.',
 'joke': 'Why do ice cubes never gossip? Because they’d rather just chill.'}

### Example-2

In [12]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import TypedDict

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define Expected Output Format
class Word(TypedDict):
    synonym: str
    antonym: str

## Use with_structured_output
llm_with_structured_output = llm.with_structured_output(Word)

## Invoke LLM
llm_with_structured_output.invoke("muscular")


{'synonym': 'brawny', 'antonym': 'scrawny'}

### Example-3

In [13]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, List, Optional, Annotated

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Create a Prompt Template with Few Shot Prompts
prompt_template = PromptTemplate(
    template="""
    Input: John here from Texas, I am 23 years old and I love playing basketball and hiking.
    Output: {{'part1': 'Texas', 'part2': 'USA', 'part3': 'Washington, D.C.'}}

    Input: Hi.. am Priya. I love singing and writing. I live in Mumbai.
    Output: {{'part1': 'Mumbai', 'part2': 'India', 'part3': 'New Delhi'}}

    Input: Friends I am Akari. I am from Yokohama. I like food.
    Output: {{'part1': 'Yokohama', 'part2': 'Japan', 'part3': 'Tokyo'}}
  
    Input: {query}""",
    input_variables=["query"],
)

## Define a custom class UserProfile using TypedDict
class UserProfile(TypedDict):    
    part1: str
    part2: str
    part3: str


## Use with_structured_output
llm_with_structured_output = llm.with_structured_output(UserProfile)

## Format Prompt
prompt = prompt_template.format(query= "I'm Jullian, 22, live in Leeds, and I like painting and reading.")

## Invoke LLM
llm_with_structured_output.invoke(prompt)


{'part1': 'Leeds', 'part2': 'United Kingdom', 'part3': 'London'}

# <font color="#800000">LangChain Output Parser</font>

## <font color="#3366FF">JSON Output Parser</font>

### Example

In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define Expected Output Format
class UserProfile(BaseModel):
    name: str
    age: int
    location: str
    interests: List[str]

## Define Parser
parser = JsonOutputParser(pydantic_object=UserProfile)

## Define Prompt Template
prompt_template = PromptTemplate(
    template="""Parse following user Profile:
    {profile}
    
    Follow below formatting instructions:
    {format_instructions}
    """,
    input_variables=["profile"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

## Format Prompt with Input
input_prompt = prompt_template.format(profile= "I am Priya, 44, live in Bangalore and I love yoga, travel, data science.")

## Invoke LLM
output = llm.invoke(input_prompt)

## Print Content of Output
print(output.content)


{"name":"Priya","age":44,"location":"Bangalore","interests":["yoga","travel","data science"]}


#### How It Works

In [15]:
## Let see the input prompt
print(input_prompt)


Parse following user Profile:
    I am Priya, 44, live in Bangalore and I love yoga, travel, data science.

    Follow below formatting instructions:
    STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} 

#### Parse Into Python Dictionary

In [16]:
## Parse Content of Output
parsed_output = parser.parse(output.content)
print(parsed_output)


{'name': 'Priya', 'age': 44, 'location': 'Bangalore', 'interests': ['yoga', 'travel', 'data science']}


## <font color="#3366FF">XML Output Parser</font>

### Example

In [18]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import XMLOutputParser
from langchain_core.prompts import PromptTemplate


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define Parser
parser = XMLOutputParser()

## Define Prompt Template
prompt_template = PromptTemplate(
    template="""Parse following user Profile:
    {profile}
    
    Follow below formatting instructions:
    {format_instructions}
    """,
    input_variables=["profile"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

## Format Prompt with Input
input_prompt = prompt_template.format(profile= "I am Priya, 44, live in Bangalore and I love yoga, travel, data science.")

## Invoke LLM
output = llm.invoke(input_prompt)

## Print Content of Output
print(output.content)


<?xml version="1.0" encoding="UTF-8"?>
<profile>
  <name>Priya</name>
  <age>44</age>
  <location>Bangalore</location>
  <interests>
    <interest>yoga</interest>
    <interest>travel</interest>
    <interest>data science</interest>
  </interests>
</profile>


## <font color="#3366FF">Comma-Separated List Output Parser</font>

### Example

In [19]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define Parser
parser = CommaSeparatedListOutputParser()

## Define Prompt Template
prompt_template = PromptTemplate(
    template="""Answer user question:
    {question}
    
    Follow below formatting instructions:
    {format_instructions}
    """,
    input_variables=["question"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

## Format Prompt with Input
input_prompt = prompt_template.format(question= "Give all continents name.")

## Invoke LLM and Print Output
output = llm.invoke(input_prompt)
print(output.content)


Africa, Antarctica, Asia, Europe, North America, South America, Australia


#### Convert to List

In [20]:
## Parse Content of Output
parsed_output = parser.parse(output.content)
print(parsed_output)


['Africa', 'Antarctica', 'Asia', 'Europe', 'North America', 'South America', 'Australia']


## <font color="#3366FF">Pydantic Output Parser</font>

In [22]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

## Define Expected Output Format
class UserProfile(BaseModel):
    name: str
    age: int
    location: str
    interests: List[str]

## Define Parser
parser = PydanticOutputParser(pydantic_object=UserProfile)

## Define Prompt Template
prompt_template = PromptTemplate(
    template="""Parse following user Profile:
    {profile}
    
    Follow below formatting instructions:
    {format_instructions}
    """,
    input_variables=["profile"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

## Format Prompt with Input
input_prompt = prompt_template.format(profile= "I am Priya, 44, live in Bangalore and I love yoga, travel, data science.")

## Invoke LLM
output = llm.invoke(input_prompt)

## Parse Content of Output
parser.parse(output.content)


UserProfile(name='Priya', age=44, location='Bangalore', interests=['yoga', 'travel', 'data science'])